# TAPG Data Extraction to Google Sheets
Notebook ini menarik data historis 1 tahun terakhir (harian) dari Yahoo Finance dan mengunggahnya ke Google Sheets Anda.

**Prasyarat:**
1. Pastikan library terinstall: `pip install yfinance pandas gspread oauth2client`
2. Letakkan file `credentials.json` (dari Google Cloud Console) di folder yang sama dengan notebook ini.
3. Bagikan link Google Sheet Anda ke email *Service Account* yang ada di dalam `credentials.json` sebagai **Editor**.

In [ ]:
import yfinance as yf
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# === KONFIGURASI ===
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1DlWT2btmazXAGRXSVmdrEe2J4niKwJiNGGTs40xOgqE/edit?gid=0#gid=0"
CREDENTIALS_FILE = "credentials.json"

# Ticker Yahoo Finance
TICKERS = {
    "TAPG": "TAPG.JK",     # Saham TAPG
    "WTI": "CL=F",         # Crude Oil WTI
    "BRENT": "BZ=F",       # Crude Oil Brent
    "EWM": "EWM",          # iShares MSCI Malaysia ETF (Proxy CPO Malaysia)
    "USD_IDR": "IDR=X"     # Kurs USD/IDR
}

PERIOD = "1y" # Tarik data 1 tahun terakhir
INTERVAL = "1d" # Data harian

In [ ]:
# === MENGAMBIL DATA DARI YFINANCE ===
print("Memulai ekstraksi data dari yfinance...")
data_frames = []

for name, ticker in TICKERS.items():
    print(f"Menarik data {name} ({ticker})...")
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    
    if df.empty:
        print(f"Peringatan: Data tidak ditemukan untuk {ticker}")
        continue
        
    # Ambil Harga Penutupan (Close). Khusus TAPG ambil Volume juga.
    if name == "TAPG":
        df = df[['Close', 'Volume']].copy()
        # Menghindari error MultiIndex pada versi yfinance terbaru
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        df.columns = [f'{name}_Close', f'{name}_Volume']
    else:
        df = df[['Close']].copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        df.columns = [f'{name}_Close']
        
    # Hapus timezone agar mudah di-merge dan diformat string nantinya
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    data_frames.append(df)

print("\nSemua data berhasil ditarik!")

In [ ]:
# === MENGGABUNGKAN (MERGE) DATA ===
print("Menggabungkan data ke dalam satu tabel...")
merged_df = data_frames[0]
for df in data_frames[1:]:
    merged_df = merged_df.join(df, how='outer')

# Forward fill (ffill) untuk mengisi data kosong akibat perbedaan hari libur pasar (misal US libur, IHSG buka)
merged_df.fillna(method='ffill', inplace=True)
# Drop baris yang masih NaN di awal periode
merged_df.dropna(inplace=True)

# Format Date menjadi string untuk Google Sheets
merged_df.reset_index(inplace=True)
merged_df['Date'] = merged_df['Date'].dt.strftime('%Y-%m-%d')

# Ubah semua NaN atau Infs yang tersisa menjadi string kosong
merged_df.fillna("", inplace=True)

print("Bentuk data akhir (Baris, Kolom):", merged_df.shape)
display(merged_df.head())

In [ ]:
# === UPLOAD KE GOOGLE SHEETS ===
print("Autentikasi dengan Google Sheets API...")
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
try:
    creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_FILE, scope)
    client = gspread.authorize(creds)
    
    # Buka spreadsheet berdasarkan URL
    sheet = client.open_by_url(GOOGLE_SHEET_URL)
    worksheet = sheet.get_worksheet(0) # Ambil tab pertama
    
    print("Membersihkan data lama di Sheet...")
    worksheet.clear()
    
    print("Mengunggah data baru...")
    # Siapkan data: gabungan header dan row
    headers = merged_df.columns.tolist()
    data = merged_df.values.tolist()
    upload_data = [headers] + data
    
    # Update sheet sekaligus (batch update) agar lebih cepat
    worksheet.update(upload_data)
    print("\nBERHASIL! Data telah masuk ke Google Sheet Anda.")
except FileNotFoundError:
    print("\n[ERROR] File 'credentials.json' tidak ditemukan!")
    print("Pastikan Anda telah mengunduhnya dari Google Cloud Console dan menaruhnya di folder yang sama dengan notebook ini.")
except gspread.exceptions.APIError as e:
    print("\n[ERROR API] Google Sheets menolak akses.")
    print("!!! Pastikan Anda SUDAH MEMBAGIKAN (Share) Google Sheet ke email Service Account sebagai Editor. !!!")
    print("Detail error:", e)
except Exception as e:
    print("\n[ERROR] Terjadi kesalahan tak terduga:", e)
